# 05. Evaluation

All models are evaluated through one standardized harness (`src/evaluate.py`),
on three test-set variants built from the project's own cleaning functions:

- **Clean:** cross-split leakage removed (41,957 examples). This is the primary metric.
- **Leaky:** leakage left in (43,018 examples). Matches the paper's evaluation condition.
- **Leaked-only:** just the 1,061 overlapping rows, isolating the memorization effect.

This notebook produces the results table and saves all predictions for
downstream analysis in `07_results_and_error_analysis.ipynb`.

In [ ]:
# Setup
GITHUB_REPO_URL = "https://github.com/iamahmadyasin/humor-intelligence.git"

!pip install -q "transformers>=4.40,<4.51" "datasets>=2.19" scipy scikit-learn

import os, sys
if not os.path.exists("humor-intelligence"):
    !git clone -q $GITHUB_REPO_URL
REPO_DIR = os.path.abspath("humor-intelligence")
!cd "$REPO_DIR" && python src/data.py
sys.path.append(os.path.join(REPO_DIR, "src"))

In [ ]:
# Model registry
MODELS = {
    "DistilBERT-128":   {"repo": "iamahmadyasin/humor-distilbert",      "max_len": 128},
    "RoBERTa-base-128": {"repo": "iamahmadyasin/humor-roberta-base",    "max_len": 128},
    "RoBERTa-large-128":{"repo": "iamahmadyasin/humor-roberta-large",   "max_len": 128},
}

In [ ]:
# Build test variants
import pandas as pd
from evaluate import build_test_variants

variants = build_test_variants(min_words=5)
clean  = variants["clean"]
leaky  = variants["leaky"]
leaked = variants["leaked_only"]

print(f"clean:  {len(clean):>7,}")
print(f"leaky:  {len(leaky):>7,}")
print(f"leaked: {len(leaked):>7,}")

## Evaluate all models

In [ ]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from evaluate import predict, compute_metrics

device = "cuda" if torch.cuda.is_available() else "cpu"
results_dir = os.path.join(REPO_DIR, "results")
os.makedirs(results_dir, exist_ok=True)

all_rows = []

for model_name, cfg in MODELS.items():
    print(f"\n{'='*60}")
    print(f"  {model_name}  ({cfg['repo']})")
    print(f"{'='*60}")

    tokenizer = AutoTokenizer.from_pretrained(cfg["repo"])
    model = AutoModelForSequenceClassification.from_pretrained(cfg["repo"])
    model.eval().to(device)

    for variant_name, df in [("clean", clean), ("leaky", leaky), ("leaked-only", leaked)]:
        preds = predict(model, tokenizer, df["joke"].tolist(),
                       max_length=cfg["max_len"], device=device)
        m = compute_metrics(df["score"].values, preds)
        all_rows.append({"model": model_name, "eval_set": variant_name, **m})

        # save predictions for downstream analysis
        if variant_name == "clean":
            pred_df = pd.DataFrame({"score": df["score"].values,
                                    "pred": preds,
                                    "joke": df["joke"].values})
            pred_df.to_csv(os.path.join(results_dir,
                           f"predictions_{model_name.lower().replace('-','_')}.csv"),
                           index=False)

    del model, tokenizer
    torch.cuda.empty_cache()

results = pd.DataFrame(all_rows)
results.to_csv(os.path.join(results_dir, "all_metrics.csv"), index=False)
print("\nSaved to results/all_metrics.csv")

## Results: clean evaluation

In [ ]:
clean_results = (results[results["eval_set"] == "clean"]
                .set_index("model")[["n", "spearman", "pearson", "rmse", "mae"]]
                .round(4))
clean_results

## Leakage impact

The paper (Weller & Seppi, 2020) evaluates on the leaked test set. The table
below compares our models under both conditions, showing the inflation from
cross-split leakage.

In [ ]:
# clean vs leaky for each model
pivot = (results[results["eval_set"].isin(["clean", "leaky"])]
         .pivot(index="model", columns="eval_set", values="spearman")
         .round(4))
pivot["inflation"] = (pivot["leaky"] - pivot["clean"]).round(4)
pivot = pivot[["clean", "leaky", "inflation"]]
pivot

## Comparison to the paper

The paper's roBERTa-large (355M params, 5 epochs) reports Spearman **0.435** /
Pearson **0.474** on their leaked test set.

On the same (leaky) evaluation condition, our RoBERTa-large scores
Spearman **0.4404** / Pearson **0.4781** exceeding the paper's numbers.
On the clean evaluation (leakage removed), the score is **0.4323**,
with the ~0.003 gap attributable to the leakage inflation the paper carries.

In [ ]:
# full comparison table
paper = pd.DataFrame([{
    "model": "roBERTa-large (paper, leaked eval)",
    "eval_set": "leaky",
    "spearman": 0.435,
    "pearson": 0.474,
    "rmse": 1.614,
}])

comparison = pd.concat([
    results[results["eval_set"].isin(["clean", "leaky"])][["model", "eval_set", "spearman", "pearson", "rmse"]],
    paper
], ignore_index=True)

comparison_pivot = comparison.pivot(index="model", columns="eval_set", values=["spearman", "pearson"])
comparison_pivot.columns = [f"{metric}_{ev}" for metric, ev in comparison_pivot.columns]
comparison_pivot = comparison_pivot.round(4)
comparison_pivot